### Installation des dépendances

Configuration adaptée du notebook `LLM setup` fourni dans le modèle de projet pour exécuter [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) et le pipeline RAG sur Google Colab.

- L'environnement Colab est configuré avec un GPU T4 et un runtime Python 3.11.
- La cellule installe les dépendances nécessaires avec `uv`, en fixant les versions de certaines bibliothèques et en ajoutant celles nécessaires au RAG.
- L'installation est ignorée lorsque PandasAI 3.0.0 et les trois bibliothèques RAG sont déjà présents. Après une première installation, un redémarrage de la session Colab est nécessaire.
- Hors Colab, l'installation est ignorée et l'environnement peut être configuré localement avec `uv sync --extra llm --extra rag`.

In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and installed_version("pandasai") != "3.0.0":
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])

    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "peft==0.21.2",
        "sentence-transformers>=6.1.0",
        "requests",
        "pandasai==3.0.0",

        # Dépendances supplémentaires pour le RAG
        "faiss-cpu>=1.15.1",
        "langchain-text-splitters>=1.1.3",
        "datasets>=4,<6",
    ]

    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

### Récupération du dépôt GitHub

Cette cellule clone le dépôt GitHub de notre équipe dans `/content/repo` afin de rendre le package `ift6758` accessible depuis Colab.

- Le clonage est effectué uniquement si le dossier cible n'existe pas déjà.
- Le dossier du dépôt est ajouté à `sys.path` pour permettre l'importation des modules Python du projet.
- Hors Colab, cette étape est ignorée.

In [2]:
import os

if IN_COLAB:
    from google.colab import userdata

    REPO = "ammou35/LNH-data-science"
    REPO_DIR = "/content/repo"
    BRANCH = "llm-code-generation"

    if not os.path.exists(REPO_DIR):
        subprocess.check_call(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO}", REPO_DIR])

    PROJECT_DIR = os.path.join(REPO_DIR, "data_science_project")
    sys.path.insert(0, PROJECT_DIR)

### Vérification de l'environnement

- Vérification de la version de Python et des principales dépendances.
- Affichage des versions utilisées pour documenter l'environnement et faciliter la reproductibilité des résultats.

In [3]:
import sys
import torch
import numpy as np
import pandas as pd
from importlib.metadata import version

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("A Python 3.11 runtime is required.")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Transformers:", version("transformers"))
print("Accelerate:", version("accelerate"))
print("PandasAI:", version("pandasai"))

Python: 3.11.13
PyTorch: 2.6.0+cu124
NumPy: 1.26.4
pandas: 2.3.3
Transformers: 5.17.0
Accelerate: 1.15.0
PandasAI: 3.0.0


### Vérification du GPU

- Si un GPU est disponible, son nom est affiché.
- Sinon, un message rappelle de sélectionner un runtime GPU T4 dans les paramètres de Google Colab.

In [4]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

GPU: Tesla T4


### Chargement du modèle Qwen

- Le modèle Qwen3-4B-Instruct-2507 est chargé à l'aide de la fonction `load_model()` du package `ift6758`.
- Lors de la première exécution, environ 8 Go de poids sont téléchargés dans le cache Hugging Face.
- Le périphérique d'exécution et le type de données numériques du modèle sont affichés après le chargement.
- Une première requête sur le hockey permet de vérifier la génération de texte avec le gabarit de conversation de Qwen.

In [5]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded on cuda:0 (torch.float16)


In [6]:
def ask_qwen(prompt, system="You are a helpful assistant.", max_new_tokens=512):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask_qwen("In one sentence, what is a hat trick in hockey?"))

In hockey, a hat trick occurs when a player scores three goals in a single game.


### Chargement des données brutes de la NHL

Cette cellule récupère les données brutes de la NHL de 2016 à 2023 avec le code d'acquisition implémenté à la tâche 1.

In [8]:
from ift6758.data.nhl import download_seasons

download_seasons(first=2016, last=2023)


2016 regular: 0game [00:00, ?game/s]

2018 regular: 0game [00:00, ?game/s]


2016 regular: 10game [00:00, 99.83game/s]

2018 regular: 13game [00:00, 125.58game/s]
2017 regular: 14game [00:00, 18.61game/s]


2019 regular: 14game [00:00, 19.21game/s]
2017 regular: 16game [00:01,  6.73game/s]


2019 regular: 16game [00:01,  6.81game/s]
2017 regular: 17game [00:02,  5.09game/s]


2019 regular: 17game [00:02,  5.14game/s]


2019 regular: 18game [00:03,  3.99game/s]
2017 regular: 18game [00:03,  3.89game/s]
2017 regular: 19game [00:03,  3.28game/s]


2019 regular: 19game [00:03,  3.30game/s]


2019 regular: 20game [00:04,  2.81game/s]
2016 regular: 20game [00:04,  3.91game/s]


2019 regular: 21game [00:04,  2.42game/s]
2017 regular: 21game [00:04,  2.42game/s]


2019 regular: 22game [00:05,  2.20game/s]
2017 regular: 22game [00:05,  2.19game/s]


2019 regular: 23game [00:06,  2.04game/s]
2017 regular: 23game [00:06,  1.99game/s]


2019 regular: 24game [00:06,  1.91game/s]
2016 regular: 25ga

HTTPError: 429 Client Error: Too Many Requests for url: https://api-web.nhle.com/v1/gamecenter/2016020027/play-by-play

### Génération de la logique de transformation de caractéristiques avec Qwen

In [ ]:
import ift6758
from pathlib import Path
from IPython.display import Markdown, display

prompt = f"""Write a Python function compute_distance_llm(df).

Requirements from our project:
- Use pandas and numpy. Include imports and a docstring.
- df is a pandas dataframe containing NHL shot and goal events.
- The dataframe includes ice coordinates in columns x and y.
- Compute the distance between each shot and the net being targeted.
- Explain any assumptions about net coordinates and shooting direction
  in the docstring.
- Preserve missing values instead of replacing them with 0 or False.
- Return the original dataframe augmented with a "distance" column.
Return only the function and imports in a Python code block.
"""

generated_code = ask_qwen(prompt, system="You write concise, correct Python code.", max_new_tokens=2000)

PACKAGE_DIR = Path(ift6758.__file__).resolve().parent
DATA_DIR = PACKAGE_DIR / "data"

display(Markdown(generated_code))

generated_code_path = PACKAGE_DIR / "llm" / "3.2_llm.py"
generated_code_path.parent.mkdir(parents=True, exist_ok=True)
generated_code = generated_code.strip().removeprefix("```python").removesuffix("```").strip()
generated_code_path.write_text(generated_code + "\n", encoding="utf-8")

### Génération du code de nettoyage des données avec Qwen

Cette cellule demande à Qwen3-4B-Instruct-2507 de générer une implémentation Python pour l'acquisition des données play-by-play pour un match de la NHL (puisque season_to_pandas ne fait que concaténer les résultats successifs obtenus avec game_to_pandas).

In [ ]:
import ift6758
from pathlib import Path
from IPython.display import Markdown, display

prompt = f"""Write a Python function game_to_pandas_llm(game).

Requirements from our project:
- Use pandas. Include imports and a docstring.
- game is a full NHL play-by-play JSON dictionary.
- Convert the game events into a pandas dataframe.
- Include only "shot-on-goal" and "goal" events.
- Use one row per event, uniquely identified by the "game_id" and "event_id" columns.
- Include "season" as the starting year of the season (e.g., 2017 for 2017-2018).
- Include "game_type" using the NHL gameType field (2 for regular season, 3 for playoffs).
- Include period, time in period, shooting team, event type,
  ice coordinates, shooter name, goalie name and shot type.
- Use rosterSpots to retrieve player names from player IDs.
- Include whether the shot was taken at an empty net.
- Use situationCode to derive the actual manpower strength
  and whether the shooting team is at even strength,
  on a power play or shorthanded.
- situationCode follows this format:
  [away goalie][away skaters][home skaters][home goalie].
- Preserve missing values instead of replacing them with 0 or False.
- Return the formatted pandas dataframe.
Return only the function and imports in a Python code block.
"""

generated_code = ask_qwen(prompt, system="You write concise, correct Python code.", max_new_tokens=2000)

PACKAGE_DIR = Path(ift6758.__file__).resolve().parent
DATA_DIR = PACKAGE_DIR / "data"

display(Markdown(generated_code))

generated_code_path = PACKAGE_DIR / "llm" / "3.2_llm.py"
generated_code_path.parent.mkdir(parents=True, exist_ok=True)
generated_code = generated_code.strip().removeprefix("```python").removesuffix("```").strip()
generated_code_path.write_text(generated_code + "\n", encoding="utf-8")

### Intégration de la fonction générée par Qwen

Cette cellule permet l'intégration du code proposé par Qwen pour son évaluation.

In [ ]:
from ift6758.data.tidy_data import game_to_pandas

# Remplacer la ligne suivante par le code généré par Qwen dans la sortie de la
# cellule précédente (en faisant attention à ne pas copier la dernière ligne) :
game_to_pandas_llm = None

### Préparation du match de référence

Cette cellule récupère les données réelles d'un match NHL connu afin de disposer d'une référence indépendante pour la comparaison.

- Le match `2017020001`, appartenant à la saison `2017-2018`, est sélectionné comme référence.
- Si son fichier JSON existe déjà, les données sont chargées localement. Sinon, elles sont téléchargées depuis l'API NHL et sauvegardées.
- L'identifiant du match et la présence d'une liste d'événements sont vérifiés.
- Le dictionnaire game contient les données de référence utilisées dans les tests suivants.

In [ ]:
import json
import requests

SEASON_ID = 2017
GAME_ID = 2017020001

raw_path = DATA_DIR / "raw" / f"{GAME_ID}.json"

if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play",
        timeout=30,
    )
    response.raise_for_status()

    downloaded = response.json()
    assert downloaded["id"] == GAME_ID
    assert isinstance(downloaded["plays"], list)

    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(downloaded), encoding="utf-8")

game = json.loads(raw_path.read_text(encoding="utf-8"))

assert game["id"] == GAME_ID
assert isinstance(game["plays"], list)

print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

### Préparation des résultats de comparaison

Cette cellule initialise les structures utilisées pour conserver les résultats des tests.

In [ ]:
functions = {
    "Manual": game_to_pandas,
    "Qwen": game_to_pandas_llm,
}

plays = game["plays"]
events = pd.json_normalize(plays)
keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])

assert len(events) == len(plays), "Flattening changed the number of events."

results = {}
dataframes = {}

### Comparaison des implémentations

Cette section compare l'implémentation manuelle de l'équipe et celle générée par Qwen à l'aide des mêmes tests.

In [ ]:
for implementation, function in functions.items():
    results[implementation] = {
        "events": None,
        "identifiers": None,
        "missing": None,
        "summary": None,
    }

    try:
        shots = function(game)
        dataframes[implementation] = shots

        # Vérification des événements conservés
        results[implementation]["events"] = False
        assert len(shots) == int(keep.sum()), "Cleaning changed the number of retained events."
        assert set(shots["event_id"]) == set(events.loc[keep, "eventId"]), "Incorrect retained events."
        results[implementation]["events"] = True

        # Vérification des identifiants
        results[implementation]["identifiers"] = False
        assert shots[["game_id", "event_id"]].notna().all().all(), "Missing event identifier."
        assert not shots.duplicated(["game_id", "event_id"]).any(), "Duplicate event identifiers."
        results[implementation]["identifiers"] = True

        # Vérification des valeurs manquantes
        results[implementation]["missing"] = False
        display(shots.isna().sum().rename("missing_count").to_frame())

        shots_with_coordinates = shots.dropna(subset=["x", "y"])

        print("Rows excluded from a coordinate-based analysis:",
              len(shots) - len(shots_with_coordinates))
        print("Rows kept in the full shots table:", len(shots))

        results[implementation]["missing"] = True

        # Résumé des tirs et des buts par équipe
        results[implementation]["summary"] = False
        summary = shots.groupby("team", dropna=False).agg(
            shot_events=("event_id", "size"),
            goals=("is_goal", "sum"),
        )

        display(summary)
        results[implementation]["summary"] = True

        # Inspection du DataFrame
        display(shots.head(10))

    except Exception as error:
        errors[implementation] = error

### Résultats de la comparaison

Cette cellule regroupe les résultats des vérifications effectuées sur les deux implémentations dans un tableau récapitulatif.

Chaque catégorie indique si les vérifications ont réussi, échoué ou n'ont pas pu être exécutées.

In [ ]:
if all(results.values()):
    print("Manual:", dataframes["Manual"].shape)
    print("Qwen:", dataframes["Qwen"].shape)

    display(dataframes["Manual"].head(10))
    display(dataframes["Qwen"].head(10))

### Résultats de la comparaison

Cette cellule présente les résultats des trois tests effectués sur l'implémentation manuelle et celle générée par Qwen, regroupés dans un tableau pour faciliter la comparaison.

Les valeurs booléennes sont remplacées par "Réussi" et "Échoué" afin de rendre les résultats plus lisibles.

In [ ]:
comparison = pd.DataFrame.from_dict(results, orient="index")

comparison = comparison.rename(columns={
    "events": "Événements",
    "identifiers": "Identifiants",
    "missing": "Valeurs manquantes",
    "summary": "Résumé par équipe",
})

comparison = comparison.replace({True: "Succès", False: "Échec", None: "Non testé"})
comparison.index.name = "Implémentation"

display(comparison)